# P452 Project 1 — Quantum Computer Simulator
**Lecturer:** Cheng Chin  
**Due:** April 13, 2026

This notebook covers the full backend development and all checkpoint questions.
Run on Google Colab with a GPU runtime for best performance.

## Setup

In [ ]:
!pip install qiskit qiskit-aer matplotlib numpy -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

# 10-qubit AerSimulator backend
simulator = AerSimulator()
print('Backend ready:', simulator.name)

## Q1.2 — Parameter Control Loop

In [ ]:
theta = np.pi  # θ = π

qc = QuantumCircuit(2, 2)
qc.ry(theta, 0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

result = simulator.run(qc, shots=1024).result()
counts = result.get_counts()
print('Counts:', counts)

qc.draw('mpl')

### Logic Check
- `Ry(π)|0⟩ = |1⟩` (rotates by π on Bloch sphere → |0⟩ flips to |1⟩)
- CNOT with control=|1⟩ flips target: q1 → |1⟩
- Final state: |11⟩ with ~100% probability
- Any wrong θ would give a different distribution → |11⟩ uniquely identifies θ=π

## Q1.3 — 10-Qubit GHZ State

In [ ]:
n = 10
qc_ghz = QuantumCircuit(n, n)
qc_ghz.h(0)
for i in range(n - 1):
    qc_ghz.cx(i, i + 1)
qc_ghz.measure(range(n), range(n))

fig = qc_ghz.draw('mpl', fold=-1)
fig.set_size_inches(16, 5)
plt.tight_layout()
plt.show()

result = simulator.run(qc_ghz, shots=1024).result()
print(result.get_counts())

## Q1.4 — Unitarity & State Recovery

In [ ]:
# Target: (1/√2)(|201⟩ + |425⟩)
# 201 = 0b0011001001, 425 = 0b0110101001 (Qiskit little-endian: q0 = bit 0)

n = 10
sv0 = np.zeros(2**n, dtype=complex)
sv0[201] = 1/np.sqrt(2)
sv0[425] = 1/np.sqrt(2)

# ── Step 1: Initial State ──
qc1 = QuantumCircuit(n)
qc1.initialize(sv0, range(n))
sv_init = Statevector(qc1)

nonzero_init = {format(i,'010b'): round(abs(a)**2,6)
                for i,a in enumerate(sv_init.data) if abs(a)>1e-6}
print('Step 1 — Initial non-zero amplitudes:')
print(nonzero_init)

In [ ]:
# ── Step 2: Apply CNOT chain ──
qc2 = QuantumCircuit(n)
qc2.initialize(sv0, range(n))
for i in range(9):
    qc2.cx(i, i+1)
sv_after = Statevector(qc2)

nonzero_after = {format(i,'010b'): round(abs(a)**2,6)
                 for i,a in enumerate(sv_after.data) if abs(a)>1e-6}
print('Step 2 — After CNOT chain:')
print(nonzero_after)

In [ ]:
# ── Step 3: Reverse the CNOT chain ──
qc3 = QuantumCircuit(n)
qc3.initialize(sv0, range(n))
for i in range(9):
    qc3.cx(i, i+1)
for i in reversed(range(9)):
    qc3.cx(i, i+1)
sv_recovered = Statevector(qc3)

nonzero_rec = {format(i,'010b'): round(abs(a)**2,6)
               for i,a in enumerate(sv_recovered.data) if abs(a)>1e-6}
print('Step 3 — Recovered state:')
print(nonzero_rec)
print('\nMatches initial?', np.allclose(
    list(nonzero_init.values()), list(nonzero_rec.values())))

### Analysis: Unitarity
- Each CNOT is unitary (U†U = I)
- The chain of CNOTs forms a unitary product U = U₈...U₁U₀
- The inverse is U† = U₀†U₁†...U₈† (since each CNOT is self-inverse, U†ᵢ = Uᵢ)
- Applying reversed sequence recovers the exact initial state → **unitarity confirmed**

## Q2.1 — Teleportation

In [ ]:
# Alice's state: |q0⟩ = (2|0⟩ + |1⟩)/√5
theta_alice = 2 * np.arccos(2 / np.sqrt(5))

qr = QuantumRegister(3, 'q')
cr = ClassicalRegister(2, 'c')
qc_tel = QuantumCircuit(qr, cr)

# Prepare Alice's qubit
qc_tel.ry(theta_alice, 0)
qc_tel.barrier(label='Alice\'s state')

# Bell State Preparation
qc_tel.h(1)
qc_tel.cx(1, 2)
qc_tel.barrier(label='Bell State Preparation')

# Bell Measurement
qc_tel.cx(0, 1)
qc_tel.h(0)
qc_tel.barrier(label='Bell Measurement')
qc_tel.measure(qr[0], cr[0])
qc_tel.measure(qr[1], cr[1])

# Bob's correction
with qc_tel.if_test((cr[1], 1)):
    qc_tel.x(2)
with qc_tel.if_test((cr[0], 1)):
    qc_tel.z(2)

qc_tel.draw('mpl', fold=20)

## Q2.2 — Long-Distance CNOT (q0 → q4)

In [ ]:
qc_ld = QuantumCircuit(5)
n_cnots = 0

def swap_adj(a, b):
    global n_cnots
    qc_ld.cx(a,b); qc_ld.cx(b,a); qc_ld.cx(a,b)
    n_cnots += 3

swap_adj(3,4); swap_adj(2,3); swap_adj(1,2)
qc_ld.cx(0,1); n_cnots += 1
swap_adj(1,2); swap_adj(2,3); swap_adj(3,4)

print(f'Total CNOTs: {n_cnots}')
qc_ld.draw('mpl')

**Answer:** 3×6 SWAPs (each 3 CNOTs) + 1 direct CNOT = **19 total CNOT gates**

## Q2.3 — Teleportation Statistics (|0⟩ input, 1024 shots)

In [ ]:
# Teleport |0⟩
qr2 = QuantumRegister(3,'q')
cr2 = ClassicalRegister(3,'c')  # measure all 3 including Bob
qc_zero = QuantumCircuit(qr2, cr2)
# Leave q0 as |0⟩
qc_zero.h(1); qc_zero.cx(1,2)
qc_zero.cx(0,1); qc_zero.h(0)
qc_zero.measure(qr2[0], cr2[0])
qc_zero.measure(qr2[1], cr2[1])
with qc_zero.if_test((cr2[1],1)): qc_zero.x(2)
with qc_zero.if_test((cr2[0],1)): qc_zero.z(2)
qc_zero.measure(qr2[2], cr2[2])

result = simulator.run(qc_zero, shots=1024).result()
counts = result.get_counts()
bob_0 = sum(v for k,v in counts.items() if k[0]=='0')  # MSB = Bob's qubit
bob_1 = sum(v for k,v in counts.items() if k[0]=='1')
print(f'Bob |0⟩: {bob_0/1024*100:.1f}%   Bob |1⟩: {bob_1/1024*100:.1f}%')

**Expected:** Bob finds |0⟩ ~100%. Any deviation is shot noise (~1/√1024 ≈ 3%).

## Q3 — Fermi-Hubbard Trotterization

Qubit layout: q0=site1↑, q1=site1↓, q2=site2↑, q3=site2↓

JW-mapped Hamiltonian:
- **Hopping:** H_J = -J/2(X₀Z₁X₂ + Y₀Z₁Y₂) for spin-up; similarly for spin-down
- **Interaction:** H_U = U/4(I - Z₀ - Z₁ + Z₀Z₁) per site

In [ ]:
def trotter_step(qc, J, U, dt):
    t = J * dt / 2

    # Spin-up hop q0↔q2 with JW Z-string on q1
    # XZX term
    qc.h(0); qc.h(2)
    qc.cx(0,1); qc.cx(2,1)
    qc.rz(-2*t, 1)
    qc.cx(0,1); qc.cx(2,1)
    qc.h(0); qc.h(2)
    # YZY term
    qc.sdg(0); qc.h(0); qc.sdg(2); qc.h(2)
    qc.cx(0,1); qc.cx(2,1)
    qc.rz(-2*t, 1)
    qc.cx(0,1); qc.cx(2,1)
    qc.h(0); qc.s(0); qc.h(2); qc.s(2)

    # Spin-down hop q1↔q3 with JW Z-string on q2
    qc.h(1); qc.h(3)
    qc.cx(1,2); qc.cx(3,2)
    qc.rz(-2*t, 2)
    qc.cx(1,2); qc.cx(3,2)
    qc.h(1); qc.h(3)
    qc.sdg(1); qc.h(1); qc.sdg(3); qc.h(3)
    qc.cx(1,2); qc.cx(3,2)
    qc.rz(-2*t, 2)
    qc.cx(1,2); qc.cx(3,2)
    qc.h(1); qc.s(1); qc.h(3); qc.s(3)

    # Interaction: site 1 (q0,q1) and site 2 (q2,q3)
    u_phase = U * dt / 2
    for q in range(4):
        qc.rz(u_phase, q)
    from qiskit.circuit.library import RZZGate
    qc.append(RZZGate(U*dt/2), [0,1])
    qc.append(RZZGate(U*dt/2), [2,3])

def make_hubbard_qc(J, U, tau, n_trotter, init_state='1000'):
    qc = QuantumCircuit(4)
    for i, bit in enumerate(reversed(init_state)):
        if bit == '1':
            qc.x(i)
    dt = tau / n_trotter
    for _ in range(n_trotter):
        trotter_step(qc, J, U, dt)
    return qc

# Q3.1: 1 Trotter step circuit
qc_hub = make_hubbard_qc(J=1.0, U=5.0, tau=0.1, n_trotter=1)
qc_hub.draw('mpl', fold=40)

## Q3.2 — Non-Interacting Dynamics (U=0, J=1)

In [ ]:
taus = np.linspace(0, np.pi, 60)
probs = []
for tau in taus:
    qc = make_hubbard_qc(J=1.0, U=0.0, tau=tau, n_trotter=10, init_state='1000')
    sv = Statevector(qc)
    # |0010⟩: q0=0,q1=0,q2=1,q3=0 → index = 0b0100 = 4
    probs.append(abs(sv.data[4])**2)

plt.figure(figsize=(8,4))
plt.plot(taus/np.pi, probs, 'b-', lw=2, label='P(Site 2 ↑)')
plt.axvline(0.5, color='orange', ls='--', label='τ=π/2')
plt.xlabel('τ/π'); plt.ylabel('Probability')
plt.title('Non-Interacting (U=0, J=1): Electron Transfer Site 1→2')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print('Transfer complete at τ =', round(taus[np.argmax(probs)],3))
print('Analytic prediction (π/2J = π/2):', round(np.pi/2, 3))

## Q3.3 — Mott Physics (U=10, J=1)

In [ ]:
taus = np.linspace(0, np.pi, 60)
p1100, p0011 = [], []
for tau in taus:
    qc = make_hubbard_qc(J=1.0, U=10.0, tau=tau, n_trotter=10, init_state='1100')
    sv = Statevector(qc)
    # |1100⟩: q0=1,q1=1,q2=0,q3=0 → index = 0b0011 = 3
    # |0011⟩: q0=0,q1=0,q2=1,q3=1 → index = 0b1100 = 12
    p1100.append(abs(sv.data[3])**2)
    p0011.append(abs(sv.data[12])**2)

plt.figure(figsize=(8,4))
plt.plot(taus/np.pi, p1100, 'b-', lw=2, label='P(|1100⟩): both at Site 1')
plt.plot(taus/np.pi, p0011, 'r-', lw=2, label='P(|0011⟩): doublon at Site 2')
plt.xlabel('τ/π'); plt.ylabel('Probability')
plt.title('Mott Physics: U=10, J=1 — Strongly Suppressed Tunneling')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

### Discussion: Mott Insulator
- At U=10 >> J=1, the energy cost to create a doublon (two electrons on one site) is U
- Tunneling is suppressed as J²/U (second-order perturbation theory)
- P(|0011⟩) remains near zero throughout — the Mott insulating state
- Compare to U=0 where full transfer occurs at τ=π/2